# CMMD final-test evaluation

Run the six cells in order after training and post-hoc evaluation. Finish the full gating entropy grid and validation gamma sweep first. `@@PROTOCOL@@` locks reference settings and one common setting per strategy selected by mean validation AUC, before test inference. Fixed-scaling endpoints provide the two unimodal controls. Existing compatible prediction caches are reused.

Patient-cluster bootstrap intervals describe image-level metrics averaged over the fitted seeds, not an ensemble. Patient-block swap tests assume exchangeable prediction blocks under the null; seed sign tests address training consistency on the fixed cohort. Both panels have separately Holm-adjusted comparisons against Vanilla and unimodal controls. These intervals do not establish external generalization or validate the existing validation-set XAI analyses. Do not tune models after inspecting test results.


In [ ]:
# @@ENV@@ — initialize TensorFlow and import the final-test evaluator
import os
import sys
from pathlib import Path

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
BASE_PATH = Path(os.environ.get("DUM_BASE", os.environ.get("ISIC24_BASE", "/mnt/disk1/home/cgiovannoni/mod-attendance")))
sys.path.insert(0, str(BASE_PATH / "src"))

import tensorflow as tf
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

import pandas as pd
from IPython.display import display
from isic24 import paths as pth
from isic24 import statistical_evaluation as se
from isic24.xai_subset import CENTRAL_SETTINGS

pth.DATASET, pth.VARIANT = "cmmd", "processed"
ART_PATH = pth.art_path()
OUTPUT_DIR = ART_PATH / "statest"


In [ ]:
# @@PROTOCOL@@ — freeze reference and validation-selected configurations before test inference
STUDIES = {
    "vanilla": "20261015-114125",
    "fixed_scaling": "20261016-234343",
    "learnable_scaling": "20261020-224124",
    "loss_regularization": "20261019-233600",
    "controlled_fusion": "20261020-063421",
    "adaptive_gating": "20261020-020650",
}
SEEDS = [8, 47, 53, 71, 90, 101]
EXPECTED_SETTINGS = {
    "vanilla": 1, "fixed_scaling": 9, "learnable_scaling": 5,
    "loss_regularization": 5, "controlled_fusion": 1, "adaptive_gating": 5,
}
protocol = se.freeze_protocol(
    {name: ART_PATH / "runs" / name / study for name, study in STUDIES.items()},
    ART_PATH, OUTPUT_DIR, SEEDS, EXPECTED_SETTINGS, CENTRAL_SETTINGS,
    n_boot=2000, n_permutations=5000, random_seed=2026,
)
selection = pd.DataFrame(protocol["evaluations"])
display(selection.groupby(["panel", "strategy", "setting_tag", "delta"])
        .agg(n_seeds=("seed", "size"), validation_auc=("selection_val_auc", "mean")))


In [ ]:
# @@TEST_PREDICTIONS@@ — validate patient separation and predict the untouched test cohort
test_frame, predictions = se.predict_test(protocol, OUTPUT_DIR)
print(f"[TEST] {len(test_frame)} images, {test_frame.patient_id.nunique()} patients")


In [ ]:
# @@STATISTICS@@ — patient-paired confidence intervals, block-swap tests and seed sign tests
results = se.analyze_test(protocol, test_frame, predictions)


In [ ]:
# @@RESULTS@@ — inspect predictive estimates and prespecified paired contrasts
display(results["summary"].round(5))
display(results["contrasts"].round(5))
print("Differences = strategy minus baseline: higher AUC and lower Brier/ECE are favorable.")
print("95% intervals cover patient sampling conditional on fitted seeds; seed SD is separate.")
print("Intervals are pointwise; p-values use Holm correction within each panel and test unit.")


In [ ]:
# @@EXPORT@@ — save final-test tables, provenance and the selected-panel confidence-interval plot
se.save_results(results, protocol, OUTPUT_DIR)
print(f"[SAVED] Final-test results: {OUTPUT_DIR}")
